# Tarea 2: Evaluación Difusa de Riesgo Crediticio

## Objetivo
Construir y analizar un sistema de inferencia difusa para estimar el **riesgo crediticio**, utilizando datos reales del dataset **German Credit Dataset** (UCI Machine Learning Repository).

El sistema debe integrar conceptos de **Lógica Difusa** vistos en la Libreta 05 (Funciones de Pertenencia, Sistemas Mamdani, TSK, Tsukamoto) para modelar la incertidumbre inherente al análisis de riesgo crediticio.

## Estructura de la Tarea
1. **Exploración y Preprocesamiento**: Cargar y analizar el dataset
2. **Definición de Variables y Etiquetas Lingüísticas**: Seleccionar 3-4 variables relevantes
3. **Funciones de Pertenencia**: Implementar conjuntos difusos similar a la Libreta 05
4. **Base de Reglas**: Diseñar 6-10 reglas lingüísticas
5. **Implementación de Sistemas**: Mamdani, TSK, Tsukamoto
6. **Comparación y Análisis**: Evaluar rendimiento de cada modelo
7. **Conclusiones**: Discusión técnica de resultados

In [1]:
# CONFIGURACIÓN INICIAL
# =====================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Configuración estética
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Reproducibilidad
np.random.seed(42)

print("Librerías cargadas exitosamente.\n")
print("=" * 70)
print("TAREA 2: EVALUACIÓN DIFUSA DE RIESGO CREDITICIO")
print("=" * 70)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Librerías cargadas exitosamente.

TAREA 2: EVALUACIÓN DIFUSA DE RIESGO CREDITICIO


## 1. Carga y Exploración del Dataset

El **German Credit Dataset** (UCI ML) contiene información de 1000 solicitantes de crédito con 20 atributos.
Objetivo: Clasificar solicitantes como "bueno" (1) o "malo" (0) en términos de riesgo crediticio.

In [2]:
# [1.1] Cargar German Credit Dataset
# URL del dataset desde UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

# Nombres de las columnas del dataset
column_names = [
    'Status_Account', 'Duration_Months', 'Credit_History', 'Purpose',
    'Credit_Amount', 'Savings_Account', 'Employment_Years', 'Installment_Rate',
    'Personal_Status', 'Other_Debtors', 'Present_Residence', 'Property',
    'Age_Years', 'Other_Installments', 'Housing', 'Existing_Credits',
    'Job', 'Dependents', 'Telephone', 'Foreign_Worker', 'Risk'
]

try:
    # Intentar cargar desde URL
    df = pd.read_csv(url, sep=' ', names=column_names)
    print("✓ Dataset cargado desde UCI Machine Learning Repository")
except:
    # Si falla, crear un dataset sintético similar
    print("⚠ No se pudo descargar. Creando dataset sintético...")
    np.random.seed(42)
    n_samples = 1000
    
    df = pd.DataFrame({
        'Duration_Months': np.random.randint(4, 72, n_samples),
        'Credit_Amount': np.random.randint(250, 18425, n_samples),
        'Age_Years': np.random.randint(18, 75, n_samples),
        'Installment_Rate': np.random.randint(1, 4, n_samples),
        'Existing_Credits': np.random.randint(1, 4, n_samples),
        'Risk': np.random.choice([0, 1], n_samples, p=[0.3, 0.7])  # 70% buenos, 30% malos
    })

print(f"\nDataset shape: {df.shape}")
print(f"\nPrimeras filas:")
print(df.head(10))

✓ Dataset cargado desde UCI Machine Learning Repository

Dataset shape: (1000, 21)

Primeras filas:
  Status_Account  Duration_Months Credit_History Purpose  Credit_Amount  \
0            A11                6            A34     A43           1169   
1            A12               48            A32     A43           5951   
2            A14               12            A34     A46           2096   
3            A11               42            A32     A42           7882   
4            A11               24            A33     A40           4870   
5            A14               36            A32     A46           9055   
6            A14               24            A32     A42           2835   
7            A12               36            A32     A41           6948   
8            A14               12            A32     A43           3059   
9            A12               30            A34     A40           5234   

  Savings_Account Employment_Years  Installment_Rate Personal_Status  \
0 

## 2. Selección y Preprocesamiento de Variables

Se seleccionan **4 variables** clave para el análisis de riesgo crediticio:
1. **Duration_Months**: Duración del crédito (meses) - entrada
2. **Credit_Amount**: Monto del crédito - entrada
3. **Age_Years**: Edad del solicitante (años) - entrada
4. **Installment_Rate**: Porcentaje de ganancias mensuales - entrada
5. **Risk**: Clasificación de riesgo (0=Malo, 1=Bueno) - salida

Estas variables son numéricas y representan factores clave en el riesgo crediticio.

In [3]:
# [2.1] Seleccionar y normalizar variables relevantes

# Seleccionar columnas numéricas clave
if 'Duration_Months' in df.columns:
    selected_features = ['Duration_Months', 'Credit_Amount', 'Age_Years', 'Installment_Rate']
    features = [col for col in selected_features if col in df.columns]
else:
    features = list(df.columns[:-1])  # Todas excepto la última (Risk)

# Extraer target
if 'Risk' in df.columns:
    target = df['Risk'].copy()
else:
    target = df.iloc[:, -1].copy()

# Crear X (características) y y (target)
X = df[features].copy()

print(f"Características seleccionadas ({len(features)}):")
for i, feat in enumerate(features, 1):
    print(f"  {i}. {feat}")

print(f"\nEstadísticas descriptivas:")
print(X.describe())

print(f"\nDistribución de riesgo:")
print(target.value_counts())
print(f"Porcentaje: {target.value_counts(normalize=True) * 100}")

# Normalizar características a [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
X_normalized = pd.DataFrame(scaler.fit_transform(X), columns=features)

print(f"\n✓ Datos normalizados a rango [0, 1]")
print(X_normalized.head(10))

Características seleccionadas (4):
  1. Duration_Months
  2. Credit_Amount
  3. Age_Years
  4. Installment_Rate

Estadísticas descriptivas:
       Duration_Months  Credit_Amount    Age_Years  Installment_Rate
count      1000.000000    1000.000000  1000.000000       1000.000000
mean         20.903000    3271.258000    35.546000          2.973000
std          12.058814    2822.736876    11.375469          1.118715
min           4.000000     250.000000    19.000000          1.000000
25%          12.000000    1365.500000    27.000000          2.000000
50%          18.000000    2319.500000    33.000000          3.000000
75%          24.000000    3972.250000    42.000000          4.000000
max          72.000000   18424.000000    75.000000          4.000000

Distribución de riesgo:
Risk
1    700
2    300
Name: count, dtype: int64
Porcentaje: Risk
1    70.0
2    30.0
Name: proportion, dtype: float64

✓ Datos normalizados a rango [0, 1]
   Duration_Months  Credit_Amount  Age_Years  Installment_

## 3. Funciones de Pertenencia (Similar a Libreta 05)

Definimos **funciones de pertenencia triangulares y trapezoidales** para cada variable de entrada.
Estas funciones modelan **etiquetas lingüísticas** (BAJO, MEDIO, ALTO) de manera suave y continua.

In [4]:
# [3.1] Definir Funciones de Pertenencia (basado en Libreta 05)

def triangular_fuzzy(x, a, b, c):
    """Función de pertenencia triangular. a <= b <= c."""
    if x <= a or x >= c:
        return 0.0
    elif a < x <= b:
        return (x - a) / (b - a)
    else:
        return (c - x) / (c - b)

def trapezoidal_fuzzy(x, a, b, c, d):
    """Función de pertenencia trapezoidal. a <= b <= c <= d."""
    if x <= a or x >= d:
        return 0.0
    elif a < x <= b:
        return (x - a) / (b - a)
    elif b < x <= c:
        return 1.0
    else:
        return (d - x) / (d - c)

def gaussian_fuzzy(x, c, w):
    """Función de pertenencia Gaussiana (suave y continua)."""
    return np.exp(-((x - c)**2) / (w**2))

print("✓ Funciones de pertenencia cargadas")

# [3.2] Definir conjuntos difusos para cada variable de entrada
# Conjuntos para Duration_Months (0-1 normalizado)
mf_duration = {
    "Corto":    lambda x: triangular_fuzzy(x, 0.0, 0.15, 0.35),
    "Medio":    lambda x: triangular_fuzzy(x, 0.2, 0.5, 0.8),
    "Largo":    lambda x: triangular_fuzzy(x, 0.65, 0.85, 1.0),
}

# Conjuntos para Credit_Amount (0-1 normalizado)
mf_amount = {
    "Bajo":     lambda x: triangular_fuzzy(x, 0.0, 0.15, 0.35),
    "Medio":    lambda x: triangular_fuzzy(x, 0.2, 0.5, 0.8),
    "Alto":     lambda x: triangular_fuzzy(x, 0.65, 0.85, 1.0),
}

# Conjuntos para Age_Years (0-1 normalizado)
mf_age = {
    "Joven":    lambda x: triangular_fuzzy(x, 0.0, 0.2, 0.4),
    "Adulto":   lambda x: triangular_fuzzy(x, 0.25, 0.5, 0.75),
    "Mayor":    lambda x: triangular_fuzzy(x, 0.6, 0.8, 1.0),
}

# Conjuntos para Installment_Rate (0-1 normalizado)
mf_rate = {
    "Bajo":     lambda x: triangular_fuzzy(x, 0.0, 0.15, 0.35),
    "Medio":    lambda x: triangular_fuzzy(x, 0.2, 0.5, 0.8),
    "Alto":     lambda x: triangular_fuzzy(x, 0.65, 0.85, 1.0),
}

# Conjuntos de salida: Riesgo
mf_risk = {
    "Muy_Bajo":  lambda x: triangular_fuzzy(x, 0.0, 0.1, 0.25),
    "Bajo":      lambda x: triangular_fuzzy(x, 0.1, 0.25, 0.4),
    "Medio":     lambda x: triangular_fuzzy(x, 0.3, 0.5, 0.7),
    "Alto":      lambda x: triangular_fuzzy(x, 0.6, 0.75, 0.9),
    "Muy_Alto":  lambda x: triangular_fuzzy(x, 0.75, 0.9, 1.0),
}

print("✓ Conjuntos difusos definidos para todas las variables")

✓ Funciones de pertenencia cargadas
✓ Conjuntos difusos definidos para todas las variables


## 4. Base de Reglas Difusas (8 Reglas)

Basado en lógica experta sobre evaluación crediticia, se diseñan las siguientes **reglas lingüísticas**:

| Regla | Condición | Consecuencia | Justificación |
|-------|-----------|--------------|---------------|
| R1 | Crédito Alto ∧ Duración Larga | Riesgo Muy Alto | Exposición prolongada a montos grandes |
| R2 | Crédito Alto ∧ Tasa Alta | Riesgo Alto | Capacidad de pago comprometida |
| R3 | Edad Joven ∧ Crédito Alto ∧ Tasa Alta | Riesgo Muy Alto | Perfil de alto riesgo |
| R4 | Edad Adulto ∧ Crédito Medio | Riesgo Medio | Perfil equilibrado |
| R5 | Crédito Bajo ∧ Tasa Baja | Riesgo Bajo | Perfil seguro |
| R6 | Duración Corta ∧ Tasa Baja | Riesgo Muy Bajo | Baja exposición |
| R7 | Edad Mayor ∧ Crédito Medio ∧ Tasa Media | Riesgo Medio | Solidez por edad, equilibrio de monto |
| R8 | Crédito Medio ∧ Tasa Media ∧ Edad Adulto | Riesgo Medio | Perfil neutral |

Estas reglas modelan el **razonamiento experto** en evaluación crediticia de forma **interpretable**.

In [5]:
# [4.1] Implementar Reglas Difusas (Estructura de datos)

rules = [
    # (Duration_label, Amount_label, Age_label, Rate_label, Risk_label)
    ("Largo",    "Alto",   None,      "Alto",   "Muy_Alto"),   # R1
    ("Medio",    "Alto",   None,      "Alto",   "Alto"),       # R2
    ("Medio",    "Alto",   "Joven",   "Alto",   "Muy_Alto"),   # R3
    ("Medio",    "Medio",  "Adulto",  "Medio",  "Medio"),      # R4
    ("Corto",    "Bajo",   None,      "Bajo",   "Bajo"),       # R5
    ("Corto",    "Bajo",   None,      "Bajo",   "Muy_Bajo"),   # R6
    ("Medio",    "Medio",  "Mayor",   "Medio",  "Medio"),      # R7
    ("Medio",    "Medio",  "Adulto",  "Medio",  "Medio"),      # R8
]

print(f"✓ Base de reglas definida con {len(rules)} reglas")
print("\nReglas:")
for i, rule in enumerate(rules, 1):
    print(f"  R{i}: {rule}")

✓ Base de reglas definida con 8 reglas

Reglas:
  R1: ('Largo', 'Alto', None, 'Alto', 'Muy_Alto')
  R2: ('Medio', 'Alto', None, 'Alto', 'Alto')
  R3: ('Medio', 'Alto', 'Joven', 'Alto', 'Muy_Alto')
  R4: ('Medio', 'Medio', 'Adulto', 'Medio', 'Medio')
  R5: ('Corto', 'Bajo', None, 'Bajo', 'Bajo')
  R6: ('Corto', 'Bajo', None, 'Bajo', 'Muy_Bajo')
  R7: ('Medio', 'Medio', 'Mayor', 'Medio', 'Medio')
  R8: ('Medio', 'Medio', 'Adulto', 'Medio', 'Medio')


## 5. Implementación de Sistemas de Inferencia

Implementaremos los **3 sistemas** vistos en la Libreta 05:
1. **Mamdani**: Fuzzificación → Inferencia → Agregación → Defuzzificación
2. **TSK (Takagi-Sugeno)**: Modelos locales lineales
3. **Tsukamoto**: Funciones monótonas invertibles

In [6]:
# [5.1] Sistema Mamdani (basado en Libreta 05)

def fuzzify(value, mf_dict):
    """
    Fuzzificación: calcula el grado de pertenencia a cada etiqueta lingüística.
    """
    return {label: func(value) for label, func in mf_dict.items()}

def mamdani_inference(duration, amount, age, rate):
    """
    Sistema de Inferencia Mamdani completo.
    Entrada: valores normalizados [0, 1]
    Salida: riesgo estimado [0, 1]
    """
    x_risk = np.linspace(0, 1, 100)
    
    # Fuzzificación
    fuzz_duration = fuzzify(duration, mf_duration)
    fuzz_amount = fuzzify(amount, mf_amount)
    fuzz_age = fuzzify(age, mf_age)
    fuzz_rate = fuzzify(rate, mf_rate)
    
    # Agregación de reglas
    agg = np.zeros_like(x_risk)
    
    for duration_label, amount_label, age_label, rate_label, risk_label in rules:
        # Calcular grado de activación de la regla
        alpha = 1.0
        
        if duration_label is not None and duration_label in fuzz_duration:
            alpha *= fuzz_duration[duration_label]
        if amount_label is not None and amount_label in fuzz_amount:
            alpha *= fuzz_amount[amount_label]
        if age_label is not None and age_label in fuzz_age:
            alpha *= fuzz_age[age_label]
        if rate_label is not None and rate_label in fuzz_rate:
            alpha *= fuzz_rate[rate_label]
        
        # Clipping del conjunto de salida
        if risk_label in mf_risk:
            clipped = np.array([min(alpha, mf_risk[risk_label](x)) for x in x_risk])
            agg = np.maximum(agg, clipped)
    
    # Defuzzificación: Centroide (COG)
    numerator = np.sum(x_risk * agg)
    denominator = np.sum(agg)
    cog = numerator / denominator if denominator > 1e-10 else 0.5
    
    return cog

print("✓ Sistema Mamdani implementado")

# Probar con algunos casos
test_cases = [
    (0.8, 0.9, 0.2, 0.8, "Riesgo Alto (Crédito alto, edad joven, tasa alta)"),
    (0.2, 0.1, 0.5, 0.2, "Riesgo Bajo (Crédito bajo, tasa baja)"),
    (0.5, 0.5, 0.5, 0.5, "Riesgo Medio (Perfil equilibrado)"),
]

print("\n--- Pruebas del Sistema Mamdani ---")
for duration, amount, age, rate, description in test_cases:
    risk = mamdani_inference(duration, amount, age, rate)
    print(f"Input: ({duration:.1f}, {amount:.1f}, {age:.1f}, {rate:.1f}) → Risk: {risk:.3f} | {description}")

✓ Sistema Mamdani implementado

--- Pruebas del Sistema Mamdani ---
Input: (0.8, 0.9, 0.2, 0.8) → Risk: 0.879 | Riesgo Alto (Crédito alto, edad joven, tasa alta)
Input: (0.2, 0.1, 0.5, 0.2) → Risk: 0.196 | Riesgo Bajo (Crédito bajo, tasa baja)
Input: (0.5, 0.5, 0.5, 0.5) → Risk: 0.500 | Riesgo Medio (Perfil equilibrado)


In [7]:
# [5.2] Sistema TSK (Takagi-Sugeno de Orden 1)

def tsk_inference(duration, amount, age, rate):
    """
    Sistema TSK de Orden 1: consecuente es modelo lineal.
    y_i = p_i * duration + q_i * amount + r_i * age + s_i * rate + c_i
    """
    # Parámetros lineales calibrados por regla (heurísticamente)
    tsk_params = [
        {"p": 0.3, "q": 0.5, "r": -0.1, "s": 0.4, "c": 0.8},  # R1: Alto riesgo
        {"p": 0.2, "q": 0.4, "r": 0.1, "s": 0.3, "c": 0.7},   # R2
        {"p": 0.35, "q": 0.45, "r": 0.2, "s": 0.35, "c": 0.85}, # R3
        {"p": 0.1, "q": 0.15, "r": 0.1, "s": 0.1, "c": 0.5},  # R4: Riesgo medio
        {"p": -0.1, "q": 0.1, "r": -0.05, "s": 0.05, "c": 0.25}, # R5: Bajo riesgo
        {"p": -0.15, "q": 0.05, "r": -0.1, "s": 0.02, "c": 0.15}, # R6: Muy bajo
        {"p": 0.05, "q": 0.1, "r": 0.15, "s": 0.08, "c": 0.45},  # R7
        {"p": 0.08, "q": 0.12, "r": 0.08, "s": 0.1, "c": 0.48},   # R8
    ]
    
    # Fuzzificación
    fuzz_duration = fuzzify(duration, mf_duration)
    fuzz_amount = fuzzify(amount, mf_amount)
    fuzz_age = fuzzify(age, mf_age)
    fuzz_rate = fuzzify(rate, mf_rate)
    
    num = 0
    den = 0
    
    for i, (duration_label, amount_label, age_label, rate_label, _) in enumerate(rules):
        # Grado de activación
        alpha = 1.0
        if duration_label and duration_label in fuzz_duration:
            alpha *= fuzz_duration[duration_label]
        if amount_label and amount_label in fuzz_amount:
            alpha *= fuzz_amount[amount_label]
        if age_label and age_label in fuzz_age:
            alpha *= fuzz_age[age_label]
        if rate_label and rate_label in fuzz_rate:
            alpha *= fuzz_rate[rate_label]
        
        # Salida local
        params = tsk_params[i]
        y_local = (params["p"] * duration + params["q"] * amount + 
                   params["r"] * age + params["s"] * rate + params["c"])
        y_local = np.clip(y_local, 0, 1)  # Asegurar rango [0, 1]
        
        num += alpha * y_local
        den += alpha
    
    # Promedio ponderado
    risk_tsk = num / den if den > 1e-10 else 0.5
    return risk_tsk

print("✓ Sistema TSK implementado")

# Pruebas del sistema TSK
print("\n--- Pruebas del Sistema TSK ---")
for duration, amount, age, rate, description in test_cases:
    risk = tsk_inference(duration, amount, age, rate)
    print(f"Input: ({duration:.1f}, {amount:.1f}, {age:.1f}, {rate:.1f}) → Risk: {risk:.3f} | {description}")

✓ Sistema TSK implementado

--- Pruebas del Sistema TSK ---
Input: (0.8, 0.9, 0.2, 0.8) → Risk: 1.000 | Riesgo Alto (Crédito alto, edad joven, tasa alta)
Input: (0.2, 0.1, 0.5, 0.2) → Risk: 0.152 | Riesgo Bajo (Crédito bajo, tasa baja)
Input: (0.5, 0.5, 0.5, 0.5) → Risk: 0.698 | Riesgo Medio (Perfil equilibrado)


In [8]:
# [5.3] Sistema Tsukamoto (Funciones monótonas invertibles)

def S(x, a, b):
    """Función S (sigmoid) monótona creciente."""
    m = (a + b) / 2
    if x <= a:
        return 0
    elif x <= m:
        return 2 * ((x - a) / (b - a)) ** 2
    elif x <= b:
        return 1 - 2 * ((x - b) / (b - a)) ** 2
    else:
        return 1

def Z(x, a, b):
    """Función Z (inversa de S) monótona decreciente."""
    return 1 - S(x, a, b)

def S_inv(alpha, a, b):
    """Inversa de la función S."""
    m = (a + b) / 2
    alpha = np.clip(alpha, 1e-6, 1 - 1e-6)
    if alpha <= 0.5:
        return a + (b - a) * np.sqrt(alpha / 2)
    else:
        return b - (b - a) * np.sqrt((1 - alpha) / 2)

def Z_inv(alpha, a, b):
    """Inversa de la función Z."""
    return S_inv(1 - alpha, a, b)

# Conjuntos monótonos de salida para Tsukamoto
tsuka_output_sets = {
    "Muy_Bajo": ("Z", 0.1, 0.3),
    "Bajo":     ("Z", 0.2, 0.4),
    "Medio":    ("S", 0.35, 0.65),
    "Alto":     ("S", 0.6, 0.8),
    "Muy_Alto": ("S", 0.7, 0.95),
}

def tsukamoto_inference(duration, amount, age, rate):
    """
    Sistema Tsukamoto: usa inversa de funciones monótonas.
    """
    # Fuzzificación
    fuzz_duration = fuzzify(duration, mf_duration)
    fuzz_amount = fuzzify(amount, mf_amount)
    fuzz_age = fuzzify(age, mf_age)
    fuzz_rate = fuzzify(rate, mf_rate)
    
    y_values = []
    alphas = []
    
    for duration_label, amount_label, age_label, rate_label, risk_label in rules:
        # Grado de activación
        alpha = 1.0
        if duration_label and duration_label in fuzz_duration:
            alpha *= fuzz_duration[duration_label]
        if amount_label and amount_label in fuzz_amount:
            alpha *= fuzz_amount[amount_label]
        if age_label and age_label in fuzz_age:
            alpha *= fuzz_age[age_label]
        if rate_label and rate_label in fuzz_rate:
            alpha *= fuzz_rate[rate_label]
        
        # Invertir el conjunto monótono
        if risk_label in tsuka_output_sets:
            kind, a, b = tsuka_output_sets[risk_label]
            y = S_inv(alpha, a, b) if kind == "S" else Z_inv(alpha, a, b)
            y = np.clip(y, 0, 1)
            y_values.append(y)
            alphas.append(alpha)
    
    # Promedio ponderado
    alphas = np.array(alphas)
    y_values = np.array(y_values)
    risk_tsuka = np.sum(alphas * y_values) / (np.sum(alphas) + 1e-10)
    
    return risk_tsuka

print("✓ Sistema Tsukamoto implementado")

# Pruebas del sistema Tsukamoto
print("\n--- Pruebas del Sistema Tsukamoto ---")
for duration, amount, age, rate, description in test_cases:
    risk = tsukamoto_inference(duration, amount, age, rate)
    print(f"Input: ({duration:.1f}, {amount:.1f}, {age:.1f}, {rate:.1f}) → Risk: {risk:.3f} | {description}")

✓ Sistema Tsukamoto implementado

--- Pruebas del Sistema Tsukamoto ---
Input: (0.8, 0.9, 0.2, 0.8) → Risk: 0.808 | Riesgo Alto (Crédito alto, edad joven, tasa alta)
Input: (0.2, 0.1, 0.5, 0.2) → Risk: 0.263 | Riesgo Bajo (Crédito bajo, tasa baja)
Input: (0.5, 0.5, 0.5, 0.5) → Risk: 0.650 | Riesgo Medio (Perfil equilibrado)


## 6. Evaluación y Comparación de Modelos

Aplicaremos los 3 sistemas a todo el dataset y compararemos su rendimiento usando métricas estándar:
- **Accuracy**: Porcentaje de clasificaciones correctas
- **Precision, Recall, F1-Score**: Métricas por clase
- **Confusion Matrix**: Matriz de confusión

In [9]:
# [6.1] Aplicar los 3 sistemas al dataset completo

print("Aplicando sistemas de inferencia al dataset...")

# Inicializar predicciones
mamdani_predictions = []
tsk_predictions = []
tsuka_predictions = []

# Iterar sobre cada muestra del dataset
for idx in range(len(X_normalized)):
    duration = X_normalized['Duration_Months'].iloc[idx] if 'Duration_Months' in X_normalized.columns else 0.5
    amount = X_normalized['Credit_Amount'].iloc[idx] if 'Credit_Amount' in X_normalized.columns else 0.5
    age = X_normalized['Age_Years'].iloc[idx] if 'Age_Years' in X_normalized.columns else 0.5
    rate = X_normalized['Installment_Rate'].iloc[idx] if 'Installment_Rate' in X_normalized.columns else 0.5
    
    # Aplicar los 3 sistemas
    mamdani_risk = mamdani_inference(duration, amount, age, rate)
    tsk_risk = tsk_inference(duration, amount, age, rate)
    tsuka_risk = tsukamoto_inference(duration, amount, age, rate)
    
    # Clasificar como "Bueno" (1) si riesgo < 0.5, "Malo" (0) si >= 0.5
    mamdani_predictions.append(0 if mamdani_risk >= 0.5 else 1)
    tsk_predictions.append(0 if tsk_risk >= 0.5 else 1)
    tsuka_predictions.append(0 if tsuka_risk >= 0.5 else 1)

# Convertir a arrays
mamdani_pred = np.array(mamdani_predictions)
tsk_pred = np.array(tsk_predictions)
tsuka_pred = np.array(tsuka_predictions)

print(f"✓ Predicciones generadas para {len(X_normalized)} muestras")

# [6.2] Calcular métricas de rendimiento

def calculate_metrics(y_true, y_pred, model_name):
    """Calcula métricas de rendimiento."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    return {
        'Model': model_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    }

# Calcular métricas para cada modelo
y_true = target.values
metrics_data = [
    calculate_metrics(y_true, mamdani_pred, 'Mamdani'),
    calculate_metrics(y_true, tsk_pred, 'TSK'),
    calculate_metrics(y_true, tsuka_pred, 'Tsukamoto'),
]

metrics_df = pd.DataFrame(metrics_data)

print("\n" + "="*70)
print("COMPARATIVA DE MODELOS")
print("="*70)
print(metrics_df.to_string(index=False))
print("="*70)

Aplicando sistemas de inferencia al dataset...
✓ Predicciones generadas para 1000 muestras


ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

## 7. Visualización de Resultados

Visualizamos las funciones de pertenencia, matrices de confusión y comparativas de rendimiento.

In [ ]:
# [7.1] Visualizar Funciones de Pertenencia

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Duration
x = np.linspace(0, 1, 200)
ax = axes[0, 0]
for label, func in mf_duration.items():
    ax.plot(x, [func(xi) for xi in x], label=label, linewidth=2)
ax.set_title('Membresías: Duración del Crédito', fontsize=12, fontweight='bold')
ax.set_xlabel('Valor normalizado [0, 1]')
ax.set_ylabel('Grado de pertenencia μ(x)')
ax.legend()
ax.grid(True, alpha=0.3)

# Amount
ax = axes[0, 1]
for label, func in mf_amount.items():
    ax.plot(x, [func(xi) for xi in x], label=label, linewidth=2)
ax.set_title('Membresías: Monto del Crédito', fontsize=12, fontweight='bold')
ax.set_xlabel('Valor normalizado [0, 1]')
ax.set_ylabel('Grado de pertenencia μ(x)')
ax.legend()
ax.grid(True, alpha=0.3)

# Age
ax = axes[1, 0]
for label, func in mf_age.items():
    ax.plot(x, [func(xi) for xi in x], label=label, linewidth=2)
ax.set_title('Membresías: Edad del Solicitante', fontsize=12, fontweight='bold')
ax.set_xlabel('Valor normalizado [0, 1]')
ax.set_ylabel('Grado de pertenencia μ(x)')
ax.legend()
ax.grid(True, alpha=0.3)

# Rate
ax = axes[1, 1]
for label, func in mf_rate.items():
    ax.plot(x, [func(xi) for xi in x], label=label, linewidth=2)
ax.set_title('Membresías: Tasa de Cuota Mensual', fontsize=12, fontweight='bold')
ax.set_xlabel('Valor normalizado [0, 1]')
ax.set_ylabel('Grado de pertenencia μ(x)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('funciones_pertenencia.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Gráfica de funciones de pertenencia guardada como 'funciones_pertenencia.png'")

In [ ]:
# [7.2] Matrices de Confusión

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models = [
    ('Mamdani', mamdani_pred),
    ('TSK', tsk_pred),
    ('Tsukamoto', tsuka_pred)
]

for idx, (model_name, predictions) in enumerate(models):
    cm = confusion_matrix(y_true, predictions)
    
    ax = axes[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
                xticklabels=['Malo (0)', 'Bueno (1)'],
                yticklabels=['Malo (0)', 'Bueno (1)'],
                cbar=False)
    ax.set_title(f'Matriz de Confusión - {model_name}', fontweight='bold')
    ax.set_ylabel('Verdadero')
    ax.set_xlabel('Predicción')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Gráfica de matrices de confusión guardada como 'confusion_matrices.png'")

In [ ]:
# [7.3] Gráfica Comparativa de Métricas

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica 1: Comparación de Accuracy
ax = axes[0]
models_names = metrics_df['Model'].values
accuracy_vals = metrics_df['Accuracy'].values
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
bars = ax.bar(models_names, accuracy_vals, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Comparación de Accuracy', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
for bar, val in zip(bars, accuracy_vals):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Gráfica 2: Comparación de todas las métricas
ax = axes[1]
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x_pos = np.arange(len(models_names))
width = 0.2

for i, metric in enumerate(metrics_to_plot):
    values = metrics_df[metric].values
    ax.bar(x_pos + i*width, values, width, label=metric, alpha=0.8)

ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Comparación de Todas las Métricas', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos + width * 1.5)
ax.set_xticklabels(models_names)
ax.set_ylim([0, 1])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('metricas_comparativas.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Gráfica de métricas comparativas guardada como 'metricas_comparativas.png'")

## 8. Análisis y Conclusiones

### Resumen de Resultados

A continuación se presentan las conclusiones técnicas basadas en los resultados observados:

#### 1. **Rendimiento de Modelos**
- **Mamdani**: Sistema clásico que integra conocimiento experto mediante agregación de conjuntos difusos.
- **TSK**: Modelo local lineal que interpola suavemente entre regiones especializadas.
- **Tsukamoto**: Usa inversión de funciones monótonas para defuzzificación puntual.

#### 2. **Interpretabilidad vs Precisión**
- La **Lógica Difusa** proporciona modelos altamente interpretables, donde cada regla tiene un significado lingüístico claro.
- Aunque los modelos pueden no alcanzar la precisión de redes neuronales profundas, ofrecen:
  - Transparencia en el proceso de decisión
  - Facilidad para auditoría en contextos regulatorios (como crédito)
  - Incorporación directa de conocimiento experto

#### 3. **Implicaciones para Evaluación de Riesgo Crediticio**
- Los sistemas difusos son especialmente útiles para:
  - Modelar la vaguedad inherente de conceptos como "riesgo", "solvencia", "perfil crediticio"
  - Capturar transiciones suaves entre niveles de riesgo
  - Adaptarse a cambios de política mediante ajuste de parámetros
  
#### 4. **Oportunidades de Mejora**
- **Optimización de parámetros**: Usar metaheurísticas (AG, PSO, DE) para ajustar centros y anchos de funciones de pertenencia
- **Sistemas multiagente**: Especializar agentes difusos por segmentos (estudiantes, autónomos, salaridos)
- **Integración con datos reales**: Calibrar reglas con base en históricos crediticios completos
- **Combinación híbrida**: Integrar lógica difusa con redes neuronales para mayor capacidad predictiva

#### 5. **Conclusión Final**
Este proyecto demuestra cómo la **Lógica Difusa** actúa como un puente entre la intuición humana y la computación automática, permitiendo construir sistemas de decisión que son simultáneamente **precisos, interpretables y adaptativos**. Para el contexto de evaluación crediticia, esta metodología oferece una alternativa valiosa a enfoques "caja negra" tradicionales.